# JN0f · The tools we use

*On-ramp 6 of 8.*

What are the actual tools doing the work here — and which do you reach for when? Five names cover almost everything in this course, and every one of them is **open-source and free**.

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN0e · One set of numbers, many pictures](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0e_charts.ipynb)  |  Next: [JN0g · Building with agents](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0g_agents.ipynb) →

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally.** On Colab it recreates the minimal layout.

In [ ]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
def _repo_ok(_here):
    """True only if a scripts/ tree exists AND housing_rules actually imports from it.
    A stale Colab extraction satisfies 'the directory exists' while being unusable, which
    previously skipped both the module refetch AND the data fetch. Anything that cannot
    import is treated as absent; under /content (a disposable Colab tree, never a real
    checkout) the broken copy is removed so the fetch below replaces it."""
    import importlib, shutil
    for _base in [_here] + list(_here.parents):
        if not (_base/'scripts'/'build_v2').exists():
            continue
        sys.path.insert(0, str(_base/'scripts'))
        try:
            for _m in [k for k in list(sys.modules)
                       if k.split('.')[0] in ('housing_rules', 's0_keys', 'cpra_dedup')]:
                del sys.modules[_m]
            importlib.invalidate_caches()
            import housing_rules  # noqa: F401  - the real test: does the package satisfy its own __init__?
            return True
        except Exception as _e:
            print(f'modules present but unusable ({type(_e).__name__}: {_e}); refetching')
            try: sys.path.remove(str(_base/'scripts'))
            except ValueError: pass
            # Remove the broken tree ONLY where it is a downloaded extraction, never a real
            # checkout: a genuine repo has .git beside scripts/. Without this removal the
            # fetch below is skipped (its own guard also only tests existence) and the stale
            # copy survives — which is precisely the bug this replaces.
            if not (_base/'.git').exists():
                shutil.rmtree(_base/'scripts', ignore_errors=True)
                print('removed the unusable scripts/ tree; it will be re-downloaded')
            return False
    return False

_have_repo = _repo_ok(_here)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


In [ ]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Point the notebook at the data

Finds the repo root, the permit feed, and the project's shared code. The two knobs near the top are all a student changes to run another city.

In [ ]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


## The toolkit, named

- **pandas** — tables in memory (the DataFrame from JN0d).
- **matplotlib** — the pictures (JN0e).
- **the `.xlsx` feed** — the raw input: the city's CPRA export, a spreadsheet with that header-row gotcha.
- **SQLite** — an entire **database** in a single file. A **database** stores tables; you query it with **SQL** (Structured Query Language). A SQL `table` is the on-disk cousin of a pandas DataFrame.
- **MapLibre** — the open-source map engine from JN0e.

Let's actually open a database. The city's submitted report (our oracle) ships as a SQLite file:

In [ ]:
import sqlite3, pandas as pd
DB = REPO_ROOT/'databases/hcd_apr_mirror_2026-06-17_fresh.db'   # the city's submitted report, as a SQLite file
con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)   # read-only
n = con.execute('SELECT COUNT(*) FROM table_a2').fetchone()[0]   # ask SQL to count the rows in one table
print('rows in table_a2 (the city APR project list):', n)

## The same question, SQL and pandas

SQL and pandas are two dialects for one idea. Here's *permits-listed per year*, asked both ways — once in SQL with `GROUP BY`, once in pandas after pulling the table in with `read_sql`. Same answer, two tools:

In [ ]:
sql_years = pd.read_sql('SELECT YEAR, COUNT(*) AS rows FROM table_a2 GROUP BY YEAR ORDER BY YEAR', con)   # the SQL way
full = pd.read_sql('SELECT * FROM table_a2', con)            # pull the whole table into pandas
pd_years = full['YEAR'].value_counts().sort_index()          # the pandas way — same question
print('--- SQL (GROUP BY) ---'); print(sql_years.to_string(index=False))
print('\n--- pandas (value_counts) ---'); print(pd_years.to_string())

In [ ]:
md(f'''Both routes report **{int(sql_years["rows"].sum()):,}** rows across **{len(sql_years)}** years — identical, because they're asking the database the same question in two languages. SQL lives *in* the data file; pandas pulls it *into* Python. You'll use both, and now you know they're the same idea wearing two coats.''')

## Datasette — a database you can click

*(forward-looking — you'll meet it for real later.)* **Datasette** turns a SQLite file like this one into a point-and-click website: browse, filter, and chart a database with **no code at all**. It's how a non-programmer explores the same data an analyst queries — and it sets up the agent workflow in JN0g.

## The open-source stack, on one screen

The map from JN0e is the showcase: **MapLibre GL JS** (open fork of Mapbox), a keyless **OpenFreeMap** Positron basemap, libraries from the public **unpkg** CDN, drawing Berkeley's ~184 tracked development projects (the curated layer the project serves as `geometry.kml`). No API key, no account, no bill — every layer is open. *That* is why anyone can run this course. (Same artifact as JN0e; here the lesson is the **tools**, there it was the **picture**. The live, draggable map is back in JN0e.)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
_asset = REPO_ROOT/'notebooks/curriculum/assets/berkeley_maplibre.jpg'
if not _asset.exists():                             # fetch the screenshot from R2 only if it isn't already here
    import urllib.request
    _asset.parent.mkdir(parents=True, exist_ok=True)
    _req = urllib.request.Request('https://raw.githubusercontent.com/blockXblock/berkeley-housing-analysis/main/notebooks/curriculum/assets/berkeley_maplibre.jpg', headers={'User-Agent':'Mozilla/5.0'})
    _asset.write_bytes(urllib.request.urlopen(_req, timeout=60).read())
display(Image(filename=str(_asset)))               # show the saved map screenshot

**Next — JN0g:** you don't wield these tools alone — you **direct an agent** that wields them for you. (The live, draggable version of this map is back in JN0e.)

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN0e · One set of numbers, many pictures](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0e_charts.ipynb)  |  Next: [JN0g · Building with agents](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0g_agents.ipynb) →